### Encoder 기반: 주어진 텍스트를 깊이있게 읽고 이해 - Bert
### DEcoder 기반: 텍스트를 창작하고 써내려감 -GPT

### BERT vs GPT 태생이 다름... 근본 차이는 Attention이 문장을 어떻게 처리(보는가)하는데 있음
- BERT(양방향) : 문장 전체를 한 번에 통째로 본다(중간 단어 맞추는 학습), 독해 분류에 최적화
- GPT(단방향) : 오직 과거와 현재만 본다 "나는 밥을" 여기까지보면 "먹는다"를 예측
    - 뒤를 미리 볼 수 없기 때문에 미래로 가버리는 특수한 장치가 있음
    - Masked Self-Attention (치팅 방지 기법)
        - 문장을 훈련할 때 문장 전체를 통째로 주면 다음 단어를 예측하지 않고 그냥 옆에 있는 정답을 컨닝(cheating)
        - 행렬 연산을 할 때 마스킹 처리해서 softmax함수를 통과하면 확률이 0이 되게 무시
    - 자기 휘귀(Autoregressive)
        - 사용자가 "옛날 옛적에" 라고 입력하면 gpt가 다음 '한' 이라고 예측하면 
        - "옛날 옛적에 한" 을 만들고 다시 다음 단어를 예측 ... 
        - 끝을 나타내는 토큰 [EOF]를 만날 때 까지 반복
        - 디코딩 전략(Decoding Strategy) : 다음 단어를 선택할 때 (확률이 높은 단어 or 가끔 랜덤하게 엉뚱한 단어를 선택?)


In [1]:
import torch
from transformers import GPT2LMHeadModel
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2",
bos_token='</s>', eos_token='</s>', unk_token='<unk>',
pad_token='<pad>', mask_token='<mask>')

model = GPT2LMHeadModel.from_pretrained('skt/kogpt2-base-v2')

c:\Users\Playdata\miniconda3\envs\p3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Playdata\miniconda3\envs\p3.10\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--skt--kogpt2-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activ

In [11]:
# 앵무새병... 1등 단어만 맹목적으로 고르는 Greedy 특성상 한 번 특정 문구 패턴이 1등 확률을 차지하기 시작하면 반복된 문장
prompt_text = "인공지능이 인간을 대체할 수 있을까?"
input_ids = tokenizer.encode(prompt_text, return_tensors='pt')
with torch.no_grad():
    greedy_output = model.generate(
        input_ids,
        max_length = 50,
        pad_token_id = tokenizer.pad_token_id
    )
generated_text = tokenizer.decode(greedy_output[0], skip_special_tokens=True)
print(generated_text)

인공지능이 인간을 대체할 수 있을까?"라며 "그런데도 인공지능은 인간을 대체하는 것이 아니라 인간을 대체하는 것이다"라며 "인공지능이 인간을 대체하는 것이 아니라 인간을 대체하는 것이다"라고 말했다.
이어 "


In [6]:
# sampling : 모조건 1등 단어만 뽑는게 아니라 주사위 굴리듯이 확률에 기반하여 가끔은 2,3등 단어도 뽑아주는 랜덤성 기법 도입
with torch.no_grad():
    sample_output = model.generate(
        input_ids,
        max_length = 50,
        do_sample = True,   # 주사위 굴리기
        top_k = 50,         # 상위 50개 중 주사위 굴리기
        top_p = 0.9,        # 누적 확률 90% 안에서
        temperature = 0.5,  # 1이면 완전 랜덤(미친 창의성) / 0, 0.1이면 완전 안전하게 확률에 기반(창작을 거의 안함)
        repetition_penalty = 1.2, # 했던 말 또하면 벌점 
        pad_token_id = tokenizer.pad_token_id
    )
creative_text = tokenizer.decode(sample_output[0], skip_special_tokens=True)
print(creative_text)


인공지능이 인간을 대체할 수 있을까?"
"그렇다면 인간이 할 일은 무엇인가?"
"인간이 할 일을 로봇이나 다른 기계가 대신 해주면 어떨까요?"
"로봇과 인간, 인간과 인간의 공존을 생각해


### gpt-2 fine_tunning
- GPT는 단순히 아무말이나 내뱉고있음... 통제 -> 파인튜닝 Q&A
- BERT 계열은 labels에 정답을 제공했지만 GPT는 문장 생성 => 내가 제공한 문장 다음에 올 단어를 예측
- 허깅페이스 라이브러리는 labels에 input_ids 전체를 주면 내부적으로 정답지를 한 칸씩 오른쪽으로 밀어서 만들어줌

In [16]:
from torch.optim import AdamW
train_text = [
    '질문 : 자연어 처리가 뭔가요? 답변 : 컴퓨터가 인간의 언어를 이해하고 분석하는 인공지능 기술입니다.',
    '질문 : 오늘 기분이 어때? 답변 : 저는 인공지능이라 기분을 느낄 수 없지만, 당신과 대화해서 기뻐요.'
]
inputs = tokenizer(train_text, padding=True, truncation=True, return_tensors='pt')
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']
optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()
epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f"epoch {epoch + 1} / {epochs} loss = {loss.item()}")
model.eval()
test_prompt = '질문: 자연어 처리가 뭔가요? 답변:'
test_ids = tokenizer.encode(test_prompt, return_tensors='pt')
with torch.no_grad():
    test_output = model.generate(
        test_ids,
        max_length = 40,
        pad_token_id = tokenizer.pad_token_id
    )
print( tokenizer.decode(test_output[0], skip_special_tokens=True) )


epoch 1 / 10 loss = 0.07415476441383362
epoch 2 / 10 loss = 0.3370322585105896
epoch 3 / 10 loss = 0.13841070234775543
epoch 4 / 10 loss = 0.09176390618085861
epoch 5 / 10 loss = 0.07057696580886841
epoch 6 / 10 loss = 0.06685681641101837
epoch 7 / 10 loss = 0.060827046632766724
epoch 8 / 10 loss = 0.02523881383240223
epoch 9 / 10 loss = 0.04043352231383324
epoch 10 / 10 loss = 0.07263060659170151
질문: 자연어 처리가 뭔가요? 답변: 컴퓨터가 인간의 언어를 이해하고 분석하는 인공지능 기술입니다.


In [17]:
model.eval()
test_prompt = '질문: 철수가 영희를 괴롭히면 어떻게 할까요? 답변:'
test_ids = tokenizer.encode(test_prompt, return_tensors='pt')
with torch.no_grad():
    test_output = model.generate(
        test_ids,
        max_length = 40,
        temperature = 0.8,
        pad_token_id = tokenizer.pad_token_id
    )
print( tokenizer.decode(test_output[0], skip_special_tokens=True))

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


질문: 철수가 영희를 괴롭히면 어떻게 할까요? 답변: 저는 인공지능이라 기분을 느낄 수 없지만, 당신과 대화해서 기뻐요.
